# PSD estimation and optimal filtering
This notebook demonstrates `citkid.signal.psd` (PSD estimation and binning)
and `citkid.signal.opt_filt` (optimal filtering) on synthetic data. It pairs
with `documents/optimal_filtering.pdf` and uses its notation: a timestream
$S_t = \sum_k a_k A_{t - \tau_k} + n_t$ with template $A_t$, amplitudes
$a_k$, and noise spectral density $\tilde{J}_\nu$.

The example is a fractional frequency shift timestream $x(t)$ with white +
$1/f$ noise and injected pulses of known shape and amplitude. We estimate the
noise PSD, build the optimal filter, recover the pulse amplitudes, and compare
the amplitude resolution to the expectation from the PSD.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from citkid.signal.psd import get_psd, bin_psd, filter_pt
from citkid.signal.opt_filt import (
    create_nsd, create_opt_filt, apply_opt_filt, get_nfft, iterate_of
)

## Synthetic noise and pulse template
Noise is generated in the frequency domain from a one-sided PSD
$J(f) = J_w \left[1 + (f_\mathrm{knee} / f)^\alpha\right]$ with random
phases. The pulse template is a unit-height double exponential.

In [ ]:
### Sampling and noise parameters
fs = 10e3 # sample frequency in Hz
dt = 1 / fs
J_white = 1e-17 # white Sxx level in 1 / Hz
f_knee = 100. # 1/f knee frequency in Hz
alpha = 1. # 1/f exponent

def J_true(f):
    """One-sided noise PSD in 1 / Hz."""
    return J_white * (1 + (f_knee / f) ** alpha)

def make_noise(n, rng):
    """Generate a noise timestream of length n with one-sided PSD J_true."""
    f = np.fft.rfftfreq(n, dt)
    J = np.zeros_like(f)
    J[1:] = J_true(f[1:])
    # E|X_k|^2 = n J / (2 dt) gives <get_psd(x)> = J
    sigma = np.sqrt(J * n / (4 * dt))
    X = sigma * (rng.normal(size = f.size) + 1j * rng.normal(size = f.size))
    return np.fft.irfft(X, n)

### Pulse template: unit height, rise and fall times
M = 200 # template length in samples
tau_rise = 0.2e-3
tau_fall = 2e-3
t_template = np.arange(M) * dt
A_t = np.exp(-t_template / tau_fall) - np.exp(-t_template / tau_rise)
A_t /= A_t.max()

### Noise-only timestream (~105 s), used to estimate the noise
rng = np.random.default_rng(0)
x_noise = make_noise(2 ** 20, rng)

## PSD estimation with `get_psd`
`get_psd(x, dt)` returns the one-sided periodogram $2 |X_k|^2 \Delta t / N$
in units$^2$/Hz (`get_frequencies = True` also returns the frequency array).
A single periodogram scatters by 100% per bin (2 degrees of freedom), so
average periodograms of segments or bin in frequency with `bin_psd`.

`bin_psd(f, data, nbins, fmin)` bins each array in the list `data` into
log-spaced bins above `fmin` (mean by default; set `statistic` to change it)
and leaves points below `fmin` unbinned. Points with `f <= 0` are dropped.
By convention the first entry of `data` is `f` itself, so the binned
frequencies come back first. `bin_psd` replaces non-array entries of the
caller's `data` list with arrays, so pass a fresh list.

In [ ]:
f_psd, psd = get_psd(x_noise, dt, get_frequencies = True)

# average periodograms of 2**14-sample segments
nseg = 2 ** 14
segs = x_noise[:len(x_noise) // nseg * nseg].reshape(-1, nseg)
f_seg = np.fft.rfftfreq(nseg, dt)
psd_seg = np.mean([get_psd(seg, dt) for seg in segs], axis = 0)

# log binning of the full-length periodogram
f_bin, psd_bin = bin_psd(f_psd, [f_psd, psd], nbins = 200, fmin = 1)

fig, ax = plt.subplots(figsize = [7, 4], layout = 'tight')
ax.plot(f_psd[1:], psd[1:], color = '0.8', label = 'get_psd, full length')
ax.plot(f_seg[1:], psd_seg[1:], label = f'mean of {len(segs)} segments')
ax.plot(f_bin, psd_bin, label = 'bin_psd')
ax.plot(f_psd[1:], J_true(f_psd[1:]), 'k--', label = 'true')
ax.set(xscale = 'log', yscale = 'log', xlabel = 'Frequency (Hz)',
       ylabel = r'$S_{xx}$ (1/Hz)', ylim = [3e-18, 1e-13])
ax.legend()
print(f'white level: {np.mean(psd_seg[f_seg > 2e3]):.3e} 1/Hz '
      f'(true {J_white:.1e})')

### Pulse tube lines with `filter_pt`
`filter_pt(f, y, n, pt_frequency)` replaces narrow windows around pulse tube
harmonics with the mean of the neighboring points. `bin_psd` calls it when
`filter_pt_n` is set (on every entry of `data` except the first, which is
assumed to be `f`). Two quirks: `filter_pt` removes harmonics
$1, \ldots, n - 1$ (not $n$), and it modifies `y` in place, so pass a copy.
Below, 25 harmonics are injected and `n = 20` removes the first 19.

In [ ]:
pt_frequency = 1.39296 # Hz, the citkid default
t_noise = np.arange(len(x_noise)) * dt
x_pt = x_noise.copy()
for i in range(1, 26):
    x_pt += 2e-8 * np.sin(2 * np.pi * i * pt_frequency * t_noise)
f_pt, psd_pt = get_psd(x_pt, dt, get_frequencies = True)

psd_pt_filt = filter_pt(f_pt, psd_pt.copy(), n = 20,
                        pt_frequency = pt_frequency)
f_bin_pt, psd_bin_pt = bin_psd(f_pt, [f_pt, psd_pt], nbins = 200, fmin = 1)
f_bin_pt, psd_bin_ptf = bin_psd(f_pt, [f_pt, psd_pt], nbins = 200, fmin = 1,
                                filter_pt_n = 20, pt_frequency = pt_frequency)

fig, axs = plt.subplots(1, 2, figsize = [12, 4], layout = 'tight')
axs[0].plot(f_pt, psd_pt, label = 'raw')
axs[0].plot(f_pt, psd_pt_filt, label = 'filter_pt(n = 20)')
axs[0].set(xlim = [0, 40], yscale = 'log', ylim = [1e-17, 1e-8])
axs[1].plot(f_bin_pt, psd_bin_pt, label = 'bin_psd')
axs[1].plot(f_bin_pt, psd_bin_ptf, label = 'bin_psd(filter_pt_n = 20)')
axs[1].plot(f_bin_pt, J_true(f_bin_pt), 'k--', label = 'true')
axs[1].set(xscale = 'log', yscale = 'log', xlim = [0.3, 5e3])
for ax in axs:
    ax.set(xlabel = 'Frequency (Hz)', ylabel = r'$S_{xx}$ (1/Hz)')
    ax.legend()

## Noise spectral density for the optimal filter
The filter uses an FFT of length `nfft = get_nfft(M)`, the next fast FFT
length $\geq 2M - 1$. `create_nsd(j, nfft)` averages one-sided periodograms
of non-overlapping length-`nfft` segments of a signal-free timestream `j`
(odd `nfft` is supported). It works in samples, not seconds:
$\tilde{J}_\nu = 2 |X_\nu|^2 / n_\mathrm{fft}$ with DC (and Nyquist, for
even `nfft`) not doubled, so $\tilde{J}_\nu = J(f_\nu) f_s$, i.e.
`SJ * dt` is in 1/Hz. (`get_psd` also doubles the DC and Nyquist bins; the
two differ only there.)

In [ ]:
nfft = get_nfft(M)
SJ = create_nsd(x_noise, nfft)
f_nfft = np.fft.rfftfreq(nfft, dt)
print(f'M = {M}, nfft = {nfft}, frequency resolution {fs / nfft:.0f} Hz')

fig, ax = plt.subplots(figsize = [7, 4], layout = 'tight')
ax.plot(f_nfft[1:], SJ[1:] * dt, 'o', ms = 3, label = 'create_nsd * dt')
ax.plot(f_nfft[1:], J_true(f_nfft[1:]), 'k--', label = 'true')
ax.set(xscale = 'log', yscale = 'log', xlabel = 'Frequency (Hz)',
       ylabel = r'$S_{xx}$ (1/Hz)')
ax.legend()

## Optimal filter
`create_opt_filt(a, SJ, nfft)` implements Eq. 5,
$\tilde{H}_\nu = (\tilde{A}^*_\nu / \tilde{J}_\nu) /
\sum_\nu \tilde{A}_\nu \tilde{A}^*_\nu / \tilde{J}_\nu$, with the sum over
the full two-sided set of FFT bins, and returns the time-domain filter $H_t$
(length `nfft`, zero lag at the center). `apply_opt_filt(s, h)` convolves
(Eq. 7) to give $\hat{a}_\tau$ for every $\tau$. A unit-height template gives
$\hat{a}_\tau = 1$, and the output peaks at the index where the pulse starts.

For stationary Gaussian noise the expected amplitude resolution is
$$\sigma_a^{-2} = 4 \int_0^{f_s/2} \frac{|\tilde{A}(f)|^2}{J(f)}\, df,$$
with $\tilde{A}(f)$ the continuous Fourier transform of the template and
$J$ the one-sided PSD. In the discrete quantities used by `create_opt_filt`,
$\sigma_a^2 = n_\mathrm{fft} / \left(4 \sum_{\nu > 0}
|\tilde{A}_\nu|^2 / \tilde{J}_\nu\right)$ (the DC term is negligible here
because of the $1/f$ noise).

In [ ]:
h = create_opt_filt(A_t, SJ, nfft)

# check normalization and alignment on a noiseless pulse
s_check = np.zeros(4 * nfft)
s_check[nfft:nfft + M] = A_t
y_check = apply_opt_filt(s_check, h)
print(f'peak {y_check.max():.6f} at index {np.argmax(y_check)} '
      f'(pulse start {nfft})')

def expected_sigma(a, SJ, nfft):
    """Optimal-filter amplitude resolution from a one-sided PSD in samples."""
    A = np.fft.rfft(a, nfft)
    return np.sqrt(nfft / (4 * np.sum(np.abs(A[1:]) ** 2 / SJ[1:])))

SJ_true = np.concatenate([[np.inf], J_true(f_nfft[1:]) * fs])
sigma_exp = expected_sigma(A_t, SJ, nfft)
print(f'expected sigma_a: {sigma_exp:.3e} (estimated PSD), '
      f'{expected_sigma(A_t, SJ_true, nfft):.3e} (true PSD)')

fig, axs = plt.subplots(1, 2, figsize = [12, 4], layout = 'tight')
axs[0].plot(t_template * 1e3, A_t)
axs[0].set(xlabel = 'Time (ms)', ylabel = 'Template $A_t$')
t_h = (np.arange(nfft) - nfft // 2) * dt
axs[1].plot(t_h * 1e3, h)
axs[1].set(xlabel = 'Lag (ms)', ylabel = 'Filter $H_t$')

## Recover injected pulses
Inject pulses at known times with three amplitudes, filter, and read
$\hat{a}_\tau$ at the known start times. For comparison, also use a
white-noise matched filter (flat `SJ`) and the raw peak height above a
pre-pulse baseline.

In [ ]:
rng = np.random.default_rng(1)
L = 2 ** 20
spacing = 1500 # samples between pulses
starts = np.arange(5000, L - 5000, spacing)
amps_levels = np.array([0.5, 1, 2]) * 1e-6
amps_true = rng.choice(amps_levels, size = starts.size)

s = make_noise(L, rng)
for start, amp in zip(starts, amps_true):
    s[start:start + M] += amp * A_t

y = apply_opt_filt(s, h)
amps_of = y[starts]

h_white = create_opt_filt(A_t, np.ones_like(SJ), nfft)
amps_white = apply_opt_filt(s, h_white)[starts]

ipk = np.argmax(A_t)
amps_peak = np.array([s[i + ipk] - np.mean(s[i - 20:i]) for i in starts])

print(f'expected sigma_a: {sigma_exp:.3e}')
print(f'{"amp":>8} {"N":>5} {"OF mean":>10} {"OF std":>10} '
      f'{"white std":>10} {"peak std":>10}')
for amp in amps_levels:
    ix = amps_true == amp
    print(f'{amp:8.1e} {ix.sum():5d} {amps_of[ix].mean():10.3e} '
          f'{amps_of[ix].std():10.3e} {amps_white[ix].std():10.3e} '
          f'{amps_peak[ix].std():10.3e}')

# noise-only resolution: filter output away from the edges
y_noise = apply_opt_filt(x_noise, h)[nfft:-nfft]
print(f'std of filtered noise-only timestream: {y_noise.std():.3e}')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize = [12, 4], layout = 'tight')
t = np.arange(L) * dt
ix = slice(starts[3] - 500, starts[8])
axs[0].plot(t[ix], s[ix], color = '0.7', label = 'S')
axs[0].plot(t[ix], y[ix], label = r'$\hat{a}_\tau$')
axs[0].plot(t[starts[3:8]], amps_true[3:8], 'kx', label = 'true')
axs[0].set(xlabel = 'Time (s)', ylabel = 'x')
axs[0].legend()

bins = np.linspace(-0.5e-6, 3e-6, 120)
axs[1].hist(amps_of, bins, histtype = 'step', label = 'optimal filter')
axs[1].hist(amps_white, bins, histtype = 'step',
            label = 'white matched filter')
xg = np.linspace(bins[0], bins[-1], 1000)
db = bins[1] - bins[0]
for amp in amps_levels:
    n = np.sum(amps_true == amp)
    g = n * db / np.sqrt(2 * np.pi) / sigma_exp
    axs[1].plot(xg, g * np.exp(-0.5 * ((xg - amp) / sigma_exp) ** 2),
                'k--', lw = 1)
axs[1].set(xlabel = 'Recovered amplitude', ylabel = 'Counts')
axs[1].legend()

## Iterative template estimation with `iterate_of`
In practice the template is not known. `iterate_of` (Section 3 of the
document) builds a template from initial start indices, then repeatedly
filters and re-finds the starts. You supply two callables:
`build_template(s, start_idx)` returns a template of fixed length `M`, and
`get_start_idx(y)` returns start indices from the filtered timestream. The
noise spectral density is estimated once from the noise timestream `j` with
`nfft = get_nfft(M)`. It returns the templates from every iteration (row 0 is
the initial template), the final filtered timestream, filter, and starts.

The start-time origin of the template is set by the initial guess: the
iteration refines the shape and removes jitter but does not remove a global
offset. The normalization in `build_template` (here, unit peak of a noisy
average) sets the amplitude scale.

In [ ]:
rng = np.random.default_rng(2)
s_it = make_noise(L, rng)
amp_it = 1e-6
for start in starts:
    s_it[start:start + M] += amp_it * A_t

def build_template(s, start_idx):
    """Average baseline-subtracted pulses and normalize to unit height."""
    pre = 20
    idx = start_idx[(start_idx >= pre) & (start_idx + M <= len(s))]
    template = np.mean([s[i:i + M] - np.mean(s[i - pre:i]) for i in idx],
                       axis = 0)
    return template / template.max()

def get_start_idx(y):
    """Find pulse starts as peaks of the filtered timestream."""
    idx, _ = find_peaks(y, height = 0.5e-6, distance = spacing // 2)
    return idx

# initial starts from peaks of the boxcar-smoothed timestream
s_smooth = np.convolve(s_it, np.ones(10) / 10, mode = 'same')
idx0, _ = find_peaks(s_smooth, height = 0.6e-6, distance = spacing // 2)
idx0 = idx0 - 5 # smoothed peak is ~5 samples after the start
print(f'{len(idx0)} initial detections ({len(starts)} pulses)')

a_iter, y_it, h_it, idx_it = iterate_of(
    s_it, x_noise, idx0, build_template, get_start_idx, N_iter = 5
)
nearest = starts[np.argmin(np.abs(idx_it[:, None] - starts), axis = 1)]
offsets = idx_it - nearest
values, counts = np.unique(offsets, return_counts = True)
print(f'{len(idx_it)} final detections; start offsets (samples): '
      f'{dict(zip(values.tolist(), counts.tolist()))}')

amps_it = y_it[idx_it]
print(f'recovered amplitude {amps_it.mean():.3e} +/- {amps_it.std():.3e} '
      f'(true {amp_it:.1e}, expected sigma {sigma_exp:.3e})')

offset = int(np.median(offsets))
fig, ax = plt.subplots(figsize = [7, 4], layout = 'tight')
for i in [0, 1, len(a_iter) - 1]:
    ax.plot(np.arange(M) + offset, a_iter[i], label = f'iteration {i}')
ax.plot(np.arange(M), A_t, 'k--', label = 'true')
ax.set(xlabel = 'Sample from true start', ylabel = 'Template',
       xlim = [-10, 120])
ax.legend()